In [ ]:
# Modules to install
# pip install scikit-clean pandas numpy sklearn

# Importo librerías

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from itertools import product
import warnings
import pickle

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

# My libraries
from filters import *
from noisers.classes import *
from noisers.funcs import *
from testFuncs import *
from noise_cv_eval import run_5cv_experiment, run_5cv_grid, run_5cv_baseline
from cleaners import CNCNOS


rs = 33

In [3]:
dataset_root = Path("dataset")
from functools import lru_cache

@lru_cache
def n_instances(ds):
    return load_dataset_df(
        dataset=ds,
        noise_type="data_base",
        split="cc",
        encoding=None,
        root=dataset_root,
    ).shape[0]

n_desired_datasets=30

keel_datasets = sorted(
    [p.name for p in (dataset_root / "data_base").iterdir() if p.is_dir()],
    key=lambda ds: (n_instances(ds), ds),
)[:n_desired_datasets]
print(f"Available datasets: {keel_datasets}\nNumber={len(keel_datasets)}")


Available datasets: ['zoo', 'hayes-roth', 'lymphography', 'iris', 'autos', 'wine', 'sonar', 'glass', 'newthyroid', 'heart', 'cleveland', 'splice', 'ecoli', 'ionosphere', 'dermatology', 'monk-2', 'led7digit', 'wdbc', 'balance', 'pima', 'vehicle', 'vowel', 'german', 'flare', 'nursery', 'contraceptive', 'yeast', 'car', 'shuttle', 'segment']
Number=30


### Display available filters

In [4]:
print_available_filters()


['AllKNN',
 'TomekLinks',
 'ENNFilter',
 'ENNProb',
 'ENNTh',
 'MultiEditFilter',
 'NCNEdit',
 'ClassificationFilter',
 'CVCFFilter',
 'EnsembleFiltering',
 'INFFC_old_wrong',
 'IterativePartitioningFilter',
 'TABPFNClassificationFilter']

# Técnicas frente a las que comparar

### Filtros basados en distancia

In [ ]:
from sklearn.linear_model import LogisticRegression


# Rejilla global de hiperparámetros
ks = [3, 5, 9]   # Número de vecinos
thresholds = [0.5, 0.7]  # Cota probabilística
n_blocks = [5, 7] 
max_iters = [3]

def build_distance_grid(ks, thresholds, n_blocks):
    '''
    Returns a list of filters, eaach with a different hyperparameter config
    '''
    distance_grid = []
    # ENN base
    for k in ks:
        distance_grid.append({
            "name": f"ENN_k{k}",
            "filter": ENNFilter(n_neighbors=k, mode="enn", n_jobs=-1),
        })

    # ENNProb y ENNProb+Th
    for k in ks:
        distance_grid.append({
            "name": f"ENNProb_k{k}",
            "filter": ENNProbFilter(n_neighbors=k, n_jobs=-1),
        }) 
        for tau in thresholds:
            distance_grid.append({
                "name": f"ENNTh_k{k}_tau{tau}",
                "filter": ENNProbFilter(n_neighbors=k, mode="th", threshold=tau, n_jobs=-1),
            })

    #NCNEdit
    for k in ks:
        distance_grid.append({
            "name": f"NCNEdit_k{k}",
            "filter": NCNEdit(n_neighbors=k, n_jobs=-1),
        })

    # Multiedit 
    for k in ks:
        for b in n_blocks:
            for mi in max_iters:
                distance_grid.append({
                    "name": f"MultiEdit_k{k}_b{b}_mi{mi}",
                    "filter": MultiEditFilter(n_neighbors=k, n_blocks=b, n_jobs=-1, max_iter=mi),
                })

    return distance_grid

# Create the distance_based_filter_list
distance_grid = build_distance_grid(ks, thresholds, n_blocks)

# Specify dataset info
noise_type = "cla_rand"
folds = (1, 2, 3, 4, 5)

all_results = []

classifier = LogisticRegression()
classifier_name ="logReg"


for noise_k in [5, 25, 50]:
    for seed in [1]:
        all_results = []

        # Prepare the path where to store the data
        out_path = Path(f"./results/proposalEvaluation/{noise_type}/{noise_k}/seed0{seed}/{classifier_name}")
        out_path.mkdir(parents=True, exist_ok=True)

        # Test filters in each dataset
        for ds_idx, dataset in enumerate(keel_datasets):
            print(ds_idx,dataset)
            # Compute baseline (no filter) results
            baseline_df = run_5cv_baseline(
                dataset=dataset,
                noise_type=noise_type,
                seed=seed,
                classifier = classifier,
                k=noise_k,
                encoding="onehot",
                test_source="noisy",
                folds=folds,
            )

            # Compute results using filters
            distance_experiments_5 = [
                {
                    "dataset": dataset,
                    "noise_type": noise_type,
                    "seed": seed,
                    "k": noise_k,
                    "filters": {cfg["name"]: cfg["filter"]},
                    "encoding": "onehot",
                    "test_source": "clean",
                    "folds": folds,
                    "summarize": True,
                    "classifier": classifier,
                    "experiment_name": f"{dataset}|{noise_type}|{noise_k}|{cfg['name']}",
                }
                for cfg in distance_grid
            ]
            classification_df, removal_df, class_summary_df, removal_summary_df = run_5cv_grid(
                distance_experiments_5
            )
            all_results.append(
                {
                    "dataset": dataset,
                    "baseline_df": baseline_df,
                    "classification_df": classification_df,
                    "removal_df": removal_df,
                    "class_summary_df": class_summary_df,
                    "removal_summary_df": removal_summary_df,
                }
            )

            with open(out_path / "dists.pkl", "wb") as f:
                pickle.dump(all_results, f)

### Filtros basados en clasificadores

In [ ]:
# Rejilla global de hiperparámetros
cvs = [5, 7, 10]
thresholds = [0.5, 0.7]
max_iters = [3]
base_estimators = [
    ("DT", DecisionTreeClassifier(criterion="entropy", splitter="best", random_state=33)),
    ("c45", c45_like),
    ("KNN1", KNeighborsClassifier(n_neighbors=3)),
    ("LR", LogisticRegression(max_iter=1000)),
]

def build_classifier_grid(cvs, thresholds):
    ''' 
    Return classifier_based_filter_list
    '''
    classifier_grid = []
    
    # CF
    for cv in cvs: 
        for est_name, est in base_estimators:
            classifier_grid.append({
                "name": f"CF_{est_name}_cv{cv}",
                "filter": ClassificationFilter(
                    estimator=est,
                    cv=cv,
                    action="remove",
                    random_state=33,
                ),
            })
    
    #CVCF y CVCFTh 
    for cv in cvs: 
        classifier_grid.append({
            "name": f"CVCF_cv{cv}",
            "filter": CVCFFilter(
                estimator=c45_like,
                cv=cv,
                vote_rule="consensus",
                action="remove",
                random_state=33,
            ),
        })
        for tau in thresholds:
            classifier_grid.append({
                "name": f"CVCFTh_cv{cv}_tau{tau}",
                "filter": CVCFFilter(
                    estimator=c45_like,
                    cv=cv,
                    vote_rule="threshold",
                    threshold=tau,
                    action="remove",
                    random_state=33,
                ),
            })

    #EF y EFTh
    for cv in cvs: 
        classifier_grid.append({
            "name": f"Ensemble_cv{cv}",
            "filter": EnsembleFiltering(
                estimators=[est for _, est in base_estimators],
                cv=cv,
                mode="consensus",
                action="remove",
                random_state=33,
            ),
        })
        for tau in thresholds:
            classifier_grid.append({
                "name": f"EnsembleTh_cv{cv}_tau{tau}",
                "filter": EnsembleFiltering(
                    estimators=[est for _, est in base_estimators],
                    cv=cv,
                    mode="threshold",
                    threshold=tau,
                    action="remove",
                    random_state=33,
                ),
            })

    #IPF
    for cv in cvs: 
        for max_iter in max_iters:
            classifier_grid.append({
                "name": f"IPF_p{cv}_consensus",
                "filter": IterativePartitioningFilter(
                    estimator=c45_like,
                    n_partitions=cv,
                    vote_rule="consensus",
                    action="remove",
                    p_stop=0.03,
                    k_patience=3,
                    max_iter=max_iter,
                    random_state=33,
                ),
            })
            classifier_grid.append({
                "name": f"IPF_p{cv}_majority",
                "filter": IterativePartitioningFilter(
                    estimator=c45_like,
                    n_partitions=cv,
                    vote_rule="majority",
                    action="remove",
                    p_stop=0.03,
                    k_patience=3,
                    max_iter=max_iter,
                    random_state=33,
                ),
            })


    #INFFC_old_wrong e INFFC_old_wrongTh
    for cv in cvs: 
        for max_iter in max_iters:
            classifier_grid.append({
                "name": f"INFFC_old_wrong_cv{cv}",
                "filter": INFFC_old_wrong(
                    estimators=[est for _, est in base_estimators],
                    cv=cv,
                    decision_rule="consensus",
                    action="remove",
                    max_iter=max_iter,
                    max_removed_frac=0.5,
                    random_state=33,
                ),
            })
            for tau in thresholds:
                classifier_grid.append({
                    "name": f"INFFC_old_wrongTh_cv{cv}_tau{tau}",
                    "filter": INFFC_old_wrong(
                        estimators=[est for _, est in base_estimators],
                        cv=cv,
                        decision_rule="threshold",
                        threshold=tau,
                        action="remove",
                        max_iter=max_iter,
                        max_removed_frac=0.5,
                        random_state=33,
                    ),
                })
    return classifier_grid
    
classifier_grid = build_classifier_grid(cvs, thresholds)

# Specify dataset info
noise_type = "cla_rand"
folds = (1, 2, 3, 4, 5)

all_results = []

classifier = LogisticRegression()
classifier_name ="logReg"


for noise_k in [5, 25, 50]:
    for seed in [1]:
        all_results = []

        # Prepare the path where to store the data
        out_pdef df_to_tblr_colored_proposed(
    df,
    caption="",
    label="tab:proposed_vs_others",
    group_breaks=None,
    table_env=True
):
    noise_labels = [c.split("_", 1)[1] for c in df.columns if c.startswith("mean_")]
    n = len(noise_labels)
    group_breaks = set(group_breaks or [])

    def fmt_mean(x):
        return "–" if pd.isna(x) else f"{x:.2f}"

    def fmt_p(x):
        return "–" if pd.isna(x) else f"{x:.2E}".replace("E+0", "E+").replace("E-0", "E-")

    def latex_escape(s):
        return (
            str(s)
            .replace("\\", r"\textbackslash{}")
            .replace("_", r"\_")
            .replace("%", r"\%")
            .replace("&", r"\&")
        )

    lines = []

    if table_env:
        lines.append(r"\begin{table}[h]")
        lines.append(r"        \centering")
        lines.append(rf"        \caption{{{caption}}}")
        lines.append(rf"        \label{{{label}}}")

    lines.append(r"   \begin{tblr}{")
    lines.append(rf"      colspec = {{l{'c'*n}c{'c'*n}}},")
    lines.append(r"      row{1,2} = {font=\bfseries},")
    lines.append(r"      hline{1,3} = {-}{},")
    lines.append(rf"      hline{{2}} = {{{2}-{1+n}}}{{1pt}},")
    lines.append(rf"      hline{{2}} = {{{3+n}-{2+2*n}}}{{1pt}},")
    lines.append(r"    }")

    lines.append(
        r"    Method"
        + rf" & \SetCell[c={n}]{{c}} Mean"
        + " &" * n
        + rf" \SetCell[c={n}]{{c}} p-value"
        + " &" * (n - 1)
        + r" \\"
    )

    lines.append(
        r"    \textit{LogReg} & " + " & ".join(noise_labels)
        + r" &  & "
        + " & ".join(noise_labels)
        + r" \\"
    )

    prev_group = None

    for _, row in df.iterrows():
        current_group = get_group(row["Filtro"])

        if prev_group is not None and current_group != prev_group:
            lines.append(r"    \hline")

        if row["Filtro"] in group_breaks:
            lines.append(r"    \hline")

        mean_vals = []
        p_vals = []

        for nl in noise_labels:
            mean_col = f"mean_{nl}"
            p_col = f"p_{nl}"

            mean = row[mean_col]
            p = row[p_col]

            mean_s = fmt_mean(mean)
            if pd.notna(mean) and pd.notna(p) and p < 0.05:
                if mean > 0:
                    mean_s = rf"\textcolor{{green!60!black}}{{{mean_s}}}"
                elif mean < 0:
                    mean_s = rf"\textcolor{{red!70!black}}{{{mean_s}}}"

            p_s = fmt_p(p)
            if pd.notna(p) and p < 0.05:
                p_s = rf"\textbf{{{p_s}}}"

            mean_vals.append(mean_s)
            p_vals.append(p_s)

        lines.append(
            f"    {latex_escape(row['Filtro'])} & "
            + " & ".join(mean_vals)
            + r" &  & "
            + " & ".join(p_vals)
            + r" \\"
        )

        prev_group = current_group

    lines.append(r"    \end{tblr}")

    if table_env:
        lines.append(r"    \end{table}")

    return "\n".join(lines)


# Dividir tabla en 3 partes
splits = np.array_split(tabla, 3)

tblrs = [
    df_to_tblr_colored_proposed(
        part,
        table_env=False
    )
    for part in splits
]

latex_3cols = (
r"""
\begin{landscape}
\begin{table}[p]
\centering
\caption{Comparación del filtro propuesto frente al resto de técnicas.}
\label{tab:proposed_vs_others}

\tiny
\setlength{\tabcolsep}{2pt}

\begin{minipage}{0.32\linewidth}
\centering
"""
+ tblrs[0] +
r"""
\end{minipage}
\hfill
\begin{minipage}{0.32\linewidth}
\centering
"""
+ tblrs[1] +
r"""
\end{minipage}
\hfill
\begin{minipage}{0.32\linewidth}
\centering
"""
+ tblrs[2] +
r"""
\end{minipage}

\end{table}
\end{landscape}
"""
)

print(latex_3cols)

latex = df_to_tblr_colored_proposed(
    tabla,
    caption="Comparación del filtro propuesto frente al resto de técnicas.",
    label="tab:proposed_vs_others"
)
print(latex)ath = Path(f"./results/proposalEvaluation/{noise_type}/{noise_k}/seed0{seed}/{classifier_name}")
        out_path.mkdir(parents=True, exist_ok=True)
                
        for ds_idx,dataset in enumerate(keel_datasets):
            print(ds_idx, dataset)
            # baseline_df = run_5cv_baseline(
            #     dataset=dataset,
            #     noise_type=noise_type,
            #     seed=seed,
            #     k=noise_k,
            #     encoding="onehot",
            #     test_source="noisy",
            #     folds=folds,
            #     classifier = classifier
            # )
            classifier_experiments_5 = [
                {
                    "dataset": dataset,
                    "noise_type": noise_type,
                    "seed": seed,
                    "k": noise_k,
                    "filters": {cfg["name"]: cfg["filter"]},
                    "encoding": "onehot",
                    "test_source": "clean",
                    "folds": folds,
                    "summarize": True,
                    "classifier": classifier,
                    "experiment_name": f"{dataset}|{noise_type}|{noise_k}|{cfg['name']}",
                }
                for cfg in classifier_grid
            ]
            classification_df, removal_df, class_summary_df, removal_summary_df = run_5cv_grid(
                classifier_experiments_5,
                save_path="results_5cv",
                save_format="pickle",
                warnings_path="results_5cv/warnings_run_5cv_grid_classif.txt",
                clear_warnings_file=False,
            )
            all_results.append({
                "dataset": dataset,
                "baseline_df": {},
                "classification_df": classification_df,
                "removal_df": removal_df,
                "class_summary_df": class_summary_df,
                "removal_summary_df": removal_summary_df,
            })

            with open(out_path / "classif.pkl", "wb") as f:
                pickle.dump(all_results, f)

### Métodos de limpieza de ruido

In [ ]:
noise_type = "cla_rand"
seed = 1
noise_k = 5
folds = (1, 2, 3, 4, 5)

ks = [3,5,9]
cvs = [5,7,9]
max_iters = [3]

# Rejilla de CNCNOS
wiper_grid = [
    {
        "name": "CNCNOS_default",
        "filter": CNCNOS(
            base_neighbors=k,
            score_neighbors=k,
            cv=cv,
            max_iter=max_iter,
            stagnation_patience=2,
            min_class_count=2,
            random_state=33,
            n_jobs=-1,
        )   
    }
    for max_iter in max_iters
    for cv in cvs
    for k in ks

]

# Specify dataset info
noise_type = "cla_rand"
folds = (1, 2, 3, 4, 5)


classifier = LogisticRegression()
classifier_name ="logReg"


for noise_k in [5, 25, 50]:
    for seed in [1]:
        all_results = []

        # Prepare the path where to store the data
        out_path = Path(f"./results/proposalEvaluation/{noise_type}/{noise_k}/seed0{seed}/{classifier_name}")
        out_path.mkdir(parents=True, exist_ok=True)
        
        for ds_idx, dataset in enumerate(keel_datasets):
            print(ds_idx, dataset)

            # baseline_df = run_5cv_baseline(
            #     dataset=dataset,
            #     noise_type=noise_type,
            #     seed=seed,
            #     k=noise_k,
            #     encoding="onehot",
            #     test_source="noisy",
            #     folds=folds,
            #     classifier=classifier,
            # )

            cnc_experiments = [
                {
                    "dataset": dataset,
                    "noise_type": noise_type,
                    "seed": seed,
                    "k": noise_k,
                    "filters": {cfg["name"]: cfg["filter"]},
                    "encoding": "onehot",
                    "test_source": "clean",
                    "folds": folds,
                    "summarize": True,
                    "classifier": classifier,
                    "experiment_name": f"{dataset}|{noise_type}|{noise_k}|{cfg['name']}",
                }
                for cfg in wiper_grid
            ]

            classification_df, removal_df, class_summary_df, removal_summary_df = run_5cv_grid(
                cnc_experiments,
                save_path="results_5cv_cncnos",
                save_format="pickle",
                warnings_path="results_5cv_cncnos/warnings_run_5cv_grid_cncnos.txt",
                clear_warnings_file=False,
            )

            all_results.append({
                "dataset": dataset,
                "baseline_df": {},
                "classification_df": classification_df,
                "removal_df": removal_df,
                "class_summary_df": class_summary_df,
                "removal_summary_df": removal_summary_df,
            })

            with open(out_path / "cncnos.pkl", "wb") as f:
                pickle.dump(all_results, f)

# TABPFN (individual)

In [ ]:
from tabpfn import TabPFNClassifier

# Set noise_type to NCAR
noise_type = "cla_rand"
# Set the classifier (to use after filtering)
classifier = LogisticRegression()
classifier_name ="logReg"
cvs = [5,7,10]
thresholds = [0.5, 0.7]

def build_tabpfn_classifier_grid(cvs, thresholds):
    grid = []

    # ClassificationFilter con TabPFN
    for cv in cvs:
        grid.append({
            "name": f"TABPFN_CF_cv{cv}",
            "filter": ClassificationFilter(
                estimator=TabPFNClassifier(device="auto", random_state=33),
                cv=cv,
                action="remove",
                random_state=33,
            ),
        })

    # CVCF con TabPFN
    for cv in cvs:
        grid.append({
            "name": f"TABPFN_CVCF_cv{cv}",
            "filter": CVCFFilter(
                estimator=TabPFNClassifier(device="auto", random_state=33),
                cv=cv,
                vote_rule="consensus",
                action="remove",
                random_state=33,
            ),
        })
        for tau in thresholds:
            grid.append({
                "name": f"TABPFN_CVCFTh_cv{cv}_tau{tau}",
                "filter": CVCFFilter(
                    estimator=TabPFNClassifier(device="auto", random_state=33),
                    cv=cv,
                    vote_rule="threshold",
                    threshold=tau,
                    action="remove",
                    random_state=33,
                ),
            })

    # IPF con TabPFN
    for cv in cvs:
        grid.append({
            "name": f"TABPFN_IPF_cv{cv}_consensus",
            "filter": IterativePartitioningFilter(
                estimator=TabPFNClassifier(device="auto", random_state=33),
                n_partitions=cv,
                vote_rule="consensus",
                action="remove",
                p_stop=0.03,
                k_patience=3,
                max_iter=3,
                random_state=33,
            ),
        })
        grid.append({
            "name": f"TABPFN_IPF_cv{cv}_majority",
            "filter": IterativePartitioningFilter(
                estimator=TabPFNClassifier(device="auto", random_state=33),
                n_partitions=cv,
                vote_rule="majority",
                action="remove",
                p_stop=0.03,
                k_patience=3,
                max_iter=3,
                random_state=33,
            ),
        })

    return grid


tabpfn_classifier_grid = build_tabpfn_classifier_grid(cvs, thresholds)


for noise_k in [5, 25, 50]:
    for seed in [1]:
        all_results = []

        # Prepare the path where to store the data
        out_path = Path(f"./results/proposalEvaluation/{noise_type}/{noise_k}/seed0{seed}/{classifier_name}")
        out_path.mkdir(parents=True, exist_ok=True)

        for ds_idx, dataset in enumerate(keel_datasets):
            print(ds_idx, dataset)
            # baseline_df = run_5cv_baseline(
            #     dataset=dataset,
            #     noise_type=noise_type,
            #     seed=seed,
            #     classifier=classifier,
            #     k=noise_k,
            #     encoding="onehot",
            #     test_source="noisy",
            #     folds=(1, 2, 3, 4, 5),
            # )

            tabpfn_experiments_5 = [
                {
                    "dataset": dataset,
                    "noise_type": noise_type,
                    "seed": seed,
                    "k": noise_k,
                    "filters": {cfg["name"]: cfg["filter"]},
                    "encoding": "onehot",
                    "test_source": "clean",
                    "folds": (1, 2, 3, 4, 5),
                    "summarize": True,
                    "classifier": classifier,
                    "experiment_name": f"{dataset}|{noise_type}|{noise_k}|{cfg['name']}",
                }
                for cfg in tabpfn_classifier_grid
            ]

            classification_df, removal_df, class_summary_df, removal_summary_df = run_5cv_grid(
                tabpfn_experiments_5
            )

            all_results.append({
                "dataset": dataset,
                "baseline_df": {},
                "classification_df": classification_df,
                "removal_df": removal_df,
                "class_summary_df": class_summary_df,
                "removal_summary_df": removal_summary_df,
            })

            with open(out_path / "TabPFN.pkl", "wb") as f:
                pickle.dump(all_results, f)

# Pruebas individuales

### CVCF replay

In [7]:
# Rejilla global de hiperparámetros
cvs = [5, 7, 10]
thresholds = [0.5, 0.7]
base_estimators = [
    ("DT", DecisionTreeClassifier(criterion="entropy", splitter="best", random_state=33)),
    ("KNN1", KNeighborsClassifier(n_neighbors=3)),
    ("LR", LogisticRegression(max_iter=1000)),
]

def build_classifier_grid(cvs, thresholds):
    ''' 
    Return classifier_based_filter_list
    '''
    classifier_grid = []

    for est_name, est in base_estimators:
        #CVCF y CVCFTh 
        for cv in cvs: 
            classifier_grid.append({
                "name": f"CVCF_cv{cv}_{est_name}",
                "filter": CVCFFilter(
                    estimator=est,
                    cv=cv,
                    vote_rule="consensus",
                    action="remove",
                    random_state=33,
                ),
            })
            for tau in thresholds:
                classifier_grid.append({
                    "name": f"CVCFTh_cv{cv}_tau{tau}_{est_name}",
                    "filter": CVCFFilter(
                        estimator=est,
                        cv=cv,
                        vote_rule="threshold",
                        threshold=tau,
                        action="remove",
                        random_state=33,
                    ),
                })
    return classifier_grid
    
classifier_grid = build_classifier_grid(cvs, thresholds)

# Specify dataset info
noise_type = "cla_rand"
folds = (1, 2, 3, 4, 5)

all_results = []

classifier = LogisticRegression()
classifier_name ="logReg"


for noise_k in [5, 25, 50]:
    for seed in [1]:
        all_results = []

        # Prepare the path where to store the data
        out_path = Path(f"./results/proposalEvaluation/{noise_type}/{noise_k}/seed0{seed}/{classifier_name}")
        out_path.mkdir(parents=True, exist_ok=True)
                
        for ds_idx,dataset in enumerate(keel_datasets):
            print(ds_idx, dataset)
            # baseline_df = run_5cv_baseline(
            #     dataset=dataset,
            #     noise_type=noise_type,
            #     seed=seed,
            #     k=noise_k,
            #     encoding="onehot",
            #     test_source="noisy",
            #     folds=folds,
            #     classifier = classifier
            # )
            classifier_experiments_5 = [
                {
                    "dataset": dataset,
                    "noise_type": noise_type,
                    "seed": seed,
                    "k": noise_k,
                    "filters": {cfg["name"]: cfg["filter"]},
                    "encoding": "onehot",
                    "test_source": "clean",
                    "folds": folds,
                    "summarize": True,
                    "classifier": classifier,
                    "experiment_name": f"{dataset}|{noise_type}|{noise_k}|{cfg['name']}",
                }
                for cfg in classifier_grid
            ]
            classification_df, removal_df, class_summary_df, removal_summary_df = run_5cv_grid(
                classifier_experiments_5,
            )
            all_results.append({
                "dataset": dataset,
                "baseline_df": {},
                "classification_df": classification_df,
                "removal_df": removal_df,
                "class_summary_df": class_summary_df,
                "removal_summary_df": removal_summary_df,
            })

            with open(out_path / "justCVCF.pkl", "wb") as f:
                pickle.dump(all_results, f)

0 zoo


5CV experiments:   0%|          | 0/27 [00:00<?, ?it/s]

/home/juamp/Documents/Master/.venv/lib/python3.14/site-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(
/home/juamp/Documents/Master/.venv/lib/python3.14/site-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(
/home/juamp/Documents/Master/.venv/lib/python3.14/site-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(
/home/juamp/Documents/Master/.venv/lib/python3.14/site-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(
/home/juamp/Documents/Master/.venv/lib/python3.14/site-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated clas

1 hayes-roth


/home/juamp/Documents/Master/.venv/lib/python3.14/site-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=10.
  warnings.warn(


...

28 shuttle


5CV experiments:   0%|          | 0/27 [00:00<?, ?it/s]

/home/juamp/Documents/Master/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/home/juamp/Documents/Master/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/home/juamp/Documents/Master/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/home/juamp/Documents/Master/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/home/juamp/Documents/Master/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in

29 segment


5CV experiments:   0%|          | 0/27 [00:00<?, ?it/s]